In [2]:
import pandas as pd

panel = pd.read_csv("/Users/nataliam/Documents/Claude/Projects/German hospitals/data/processed/panel.csv")

print(panel.shape)
print(panel["year"].value_counts().sort_index())


(1597, 12)
year
2011    400
2015    400
2019    399
2023    398
Name: count, dtype: int64


In [3]:
print(panel.head())

   kreis_code                                  kreis_name  pop_total  \
0        1001                 Flensburg, kreisfreie Stadt    96431.0   
1        1002                      Kiel, kreisfreie Stadt   251751.0   
2        1003        Lübeck, kreisfreie Stadt, Hansestadt   217061.0   
3        1004                Neumünster, kreisfreie Stadt    79461.0   
4        1051                         Dithmarschen, Kreis   133514.0   

   pop_65_plus  share_65_plus  n_hospitals  hospitals_per_100k  aging_rank  \
0      19299.0       0.200133            5            5.185055    0.113065   
1      48598.0       0.193040           12            4.766615    0.067839   
2      50667.0       0.233423            9            4.146300    0.562814   
3      18230.0       0.229421            3            3.775437    0.500000   
4      34524.0       0.258580            2            1.497970    0.798995   

   access_rank  problem_score      category  year  
0     0.070352       0.183417  Other Kreise  2

LOOKING AT HOSPITAL ACCESS CHANGE OVER TIME: SINCE 2011 TO 2023

In [4]:
wide = panel.pivot(
    index="kreis_code",
    columns="year",
    values=["share_65_plus", "hospitals_per_100k"],
)
# flattening the multi-index column names
wide.columns = [f"{metric}_{year}" for metric, year in wide.columns]
wide = wide.reset_index()

# Computing change in access from 2011 → 2023
wide["access_change"] = (
    wide["hospitals_per_100k_2023"] - wide["hospitals_per_100k_2011"]
)

# Carrying the kreis_name forward for hover/labels
names = panel[["kreis_code", "kreis_name"]].drop_duplicates("kreis_code")
wide = wide.merge(names, on="kreis_code")

print(wide.shape)                       
print(wide["access_change"].describe()) # is the average negative? by how much?

(401, 11)
count    397.000000
mean       0.254411
std        0.987652
min       -4.230324
25%       -0.224806
50%        0.023160
75%        0.713004
max        4.894148
Name: access_change, dtype: float64


In [25]:
import plotly.express as px

# the most concerning combination: was aging in 2011 AND lost access
wide["was_aging_in_2011"] = wide["share_65_plus_2011"] > wide["share_65_plus_2011"].median()
wide["lost_access"]       = wide["access_change"] < 0
wide["aging_and_losing"]  = wide["was_aging_in_2011"] & wide["lost_access"]

fig = px.scatter(
    wide,
    x="share_65_plus_2011",
    y="access_change",
    color="aging_and_losing",
    color_discrete_map={True: "#C8385C", False: "#9AA5D9"},
    hover_name="kreis_name",
    trendline="ols",
    trendline_scope="overall", 
    trendline_color_override="gray",
    title="Change in hospital access (2011 → 2023) vs. aging in 2011",
    labels={
        "share_65_plus_2011": "Share of population aged 65+ in 2011",
        "access_change":      "Change in hospitals per 100k (2011 → 2023)",
        "aging_and_losing":   "Aging Kreis losing access",
    },
    opacity=0.7,
)

# A horizontal line at y=0 makes "lost" vs "gained" easy to see
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)

# Removing the default title
fig.update_layout(title=None, margin=dict(t=40, b=100, l=60, r=200))

# New title — inside the plot
fig.add_annotation(
    text="<b>Change in hospital access (2011 → 2023) vs. aging in 2011</b>",
    xref="paper", yref="paper",
    x=0.45, y=1.07,#0.937,
    xanchor="center", yanchor="top",
    showarrow=False,
    font=dict(size=15, color="#191240"),
)

# Commentary box 
fig.add_annotation(
    text=("The typical German Kreis hardly changed<br>"
          "in per-capita hospital access between 2011 and 2023.<br>"
          "<i>The variance, not the average, is the story.</i>"),
    xref="paper", yref="paper",
    x=0.76, y=0.84,
    xanchor="center", yanchor="top",
    showarrow=False,
    font=dict(size=11, color="#555"),
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="lightgray",
    borderwidth=1,
    borderpad=8,
)

# Source — small gray text under the chart
fig.add_annotation(
    text=("Source: Destatis Krankenhausverzeichnis· "
          "Regionalstatistik table 12411-03-03-4.<br>"
          "Hamburg and Berlin excluded due to Kreis-granularity mismatch."),
    xref="paper", yref="paper",
    x=0, y=-0.25,
    xanchor="left",
    showarrow=False,
    font=dict(size=10, color="gray"),
)


fig.show()



Checking which Kreise got MOST/LEAST HOSPITAL ACCESS throoughout the years

In [27]:
def show_extremes(panel, year, n=5):
    """Print the n Kreise with most and least hospital access for a given year."""
    year_data = panel[panel["year"] == year]
    
    cols = ["kreis_name", "kreis_code", "share_65_plus",
            "n_hospitals", "hospitals_per_100k"]
    
    top    = year_data.nlargest(n,  "hospitals_per_100k")[cols]
    bottom = year_data.nsmallest(n, "hospitals_per_100k")[cols]
    
    print(f"=== Most hospital access — {year} ===")
    print(top.to_string(index=False))
    
    print(f"\n=== Least hospital access — {year} ===")
    print(bottom.to_string(index=False))
    print()
    
    return top, bottom




In [28]:
for year in [2011, 2015, 2019, 2023]:
    show_extremes(panel, year, n=5)

=== Most hospital access — 2011 ===
                             kreis_name  kreis_code  share_65_plus  n_hospitals  hospitals_per_100k
                Baden-Baden, Stadtkreis        8211       0.266004            5            9.537616
        Berchtesgadener Land, Landkreis        9172       0.229590            8            7.900220
              Ansbach, kreisfreie Stadt        9561       0.216136            3            7.575758
                      Uelzen, Landkreis        3360       0.240524            7            7.503966
      Garmisch-Partenkirchen, Landkreis        9180       0.245139            6            7.123014

=== Least hospital access — 2011 ===
                       kreis_name  kreis_code  share_65_plus  n_hospitals  hospitals_per_100k
                 Kusel, Landkreis        7336       0.216539            0             0.00000
                Rhein-Pfalz-Kreis        7338       0.212948            0             0.00000
                 Fürth, Landkreis        957